# HotelEvaluationState

Snapshot and restore the exact Delta versions of every table reachable through governed hotel actions.

In [ ]:
action = "snapshot"
snapshot_id = "assert-run"
workspace_id = "00000000-0000-0000-0000-000000000000"
hotel_operations_lakehouse_id = "00000000-0000-0000-0000-000000000000"
hotel_guest_services_lakehouse_id = "00000000-0000-0000-0000-000000000000"

In [ ]:
from __future__ import annotations

import hashlib
import json
import re
from datetime import datetime, timezone

import notebookutils
from delta.tables import DeltaTable
from pyspark.sql import DataFrame, functions as F

FABRIC_TABLES = {
    "operations": ["maintenance_requests", "work_orders"],
    "guest_services": [
        "reservations",
        "restaurant_reservations",
        "billing_disputes",
        "special_requests",
        "action_confirmations",
        "idempotency_records",
        "audit_events",
        "action_sagas",
    ],
}


def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def lakehouse_root(lakehouse_id: str) -> str:
    return f"abfss://{workspace_id}@onelake.dfs.fabric.microsoft.com/{lakehouse_id}"


def table_path(lakehouse_id: str, table_name: str) -> str:
    return f"{lakehouse_root(lakehouse_id)}/Tables/{table_name}"


def state_root() -> str:
    return f"{lakehouse_root(hotel_guest_services_lakehouse_id)}/Files/assert-evaluation-state"


def lock_path() -> str:
    return f"{state_root()}/active.json"


def manifest_path() -> str:
    return f"{state_root()}/snapshots/{snapshot_id}.json"


def read_json(path: str) -> dict[str, object]:
    value = json.loads(notebookutils.fs.head(path, 1024 * 1024))
    if not isinstance(value, dict):
        raise RuntimeError(f"Expected a JSON object at {path}")
    return value


def remove_required(path: str) -> None:
    removed = notebookutils.fs.rm(path, False)
    if removed is not True or notebookutils.fs.exists(path):
        raise RuntimeError(f"Could not remove required evaluation state path: {path}")


def fingerprint(df: DataFrame) -> dict[str, object]:
    row_json = F.to_json(F.struct(*[F.col(name) for name in df.columns]))
    hashes = df.select(F.xxhash64(row_json).alias("row_hash"))
    aggregate = hashes.agg(
        F.count("*").alias("row_count"),
        F.sum(F.col("row_hash").cast("decimal(38,0)")).alias("hash_sum"),
        F.min("row_hash").alias("hash_min"),
        F.max("row_hash").alias("hash_max"),
    ).first()
    summary = {
        "row_count": int(aggregate["row_count"]),
        "hash_sum": str(aggregate["hash_sum"] or 0),
        "hash_min": str(aggregate["hash_min"] or 0),
        "hash_max": str(aggregate["hash_max"] or 0),
        "schema": df.schema.json(),
    }
    encoded = json.dumps(summary, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return {
        "row_count": summary["row_count"],
        "content_sha256": hashlib.sha256(encoded).hexdigest(),
    }


def table_entries() -> list[dict[str, object]]:
    lakehouses = {
        "operations": hotel_operations_lakehouse_id,
        "guest_services": hotel_guest_services_lakehouse_id,
    }
    entries: list[dict[str, object]] = []
    for lakehouse_name, table_names in FABRIC_TABLES.items():
        lakehouse_id = lakehouses[lakehouse_name]
        for table_name in table_names:
            try:
                path = table_path(lakehouse_id, table_name)
                delta = DeltaTable.forPath(spark, path)
                version_row = delta.history(1).select("version").first()
                if version_row is None:
                    raise RuntimeError("No Delta history exists")
                entries.append(
                    {
                        "lakehouse": lakehouse_name,
                        "lakehouse_id": lakehouse_id,
                        "table": table_name,
                        "version": int(version_row["version"]),
                        "fingerprint": fingerprint(delta.toDF()),
                    }
                )
            except Exception as exc:
                raise RuntimeError(
                    f"Could not snapshot {lakehouse_name}.{table_name}: "
                    f"{type(exc).__name__}: {exc}"
                ) from exc
    return entries


In [ ]:
normalized_action = action.strip().lower()
if normalized_action not in {"snapshot", "seal", "restore"}:
    raise ValueError("action must be snapshot, seal, or restore")
if re.fullmatch(r"[A-Za-z0-9_-]{1,80}", snapshot_id) is None:
    raise ValueError("snapshot_id must contain only letters, numbers, underscores, or hyphens")

if normalized_action == "snapshot":
    lock_acquired = False
    try:
        notebookutils.fs.mkdirs(state_root())
        notebookutils.fs.mkdirs(f"{state_root()}/snapshots")
        lock = {"snapshot_id": snapshot_id, "status": "active", "created_at": utc_now()}
        created = notebookutils.fs.put(lock_path(), json.dumps(lock, sort_keys=True), False)
        if created is not True:
            raise RuntimeError("Could not acquire the exclusive Fabric evaluation lock")
        lock_acquired = True
        active = read_json(lock_path())
        if active.get("snapshot_id") != snapshot_id or active.get("status") != "active":
            raise RuntimeError("Fabric evaluation lock ownership verification failed")
        manifest = {
            "schema_version": 1,
            "snapshot_id": snapshot_id,
            "created_at": utc_now(),
            "tables": table_entries(),
        }
        manifest_created = notebookutils.fs.put(
            manifest_path(), json.dumps(manifest, sort_keys=True), False
        )
        confirmed_manifest = read_json(manifest_path())
        if manifest_created is not True or confirmed_manifest.get("snapshot_id") != snapshot_id:
            raise RuntimeError("Could not persist the Fabric evaluation snapshot manifest")
        result = {
            "action": "snapshot",
            "snapshot_id": snapshot_id,
            "table_count": len(manifest["tables"]),
            "tables": manifest["tables"],
        }
    except Exception as exc:
        if notebookutils.fs.exists(manifest_path()):
            remove_required(manifest_path())
        if lock_acquired and notebookutils.fs.exists(lock_path()):
            remove_required(lock_path())
        result = {
            "action": "snapshot",
            "snapshot_id": snapshot_id,
            "table_count": 0,
            "error": f"{type(exc).__name__}: {exc}",
        }
elif normalized_action == "seal":
    if not notebookutils.fs.exists(lock_path()):
        raise RuntimeError("No active Fabric evaluation snapshot lock exists")
    active = read_json(lock_path())
    if active.get("snapshot_id") != snapshot_id or active.get("status") != "active":
        raise RuntimeError(f"Snapshot {snapshot_id} does not own an active evaluation lock")
    active["status"] = "sealed"
    active["sealed_at"] = utc_now()
    updated = notebookutils.fs.put(lock_path(), json.dumps(active, sort_keys=True), True)
    confirmed = read_json(lock_path())
    if updated is not True or confirmed.get("status") != "sealed":
        raise RuntimeError("Could not seal the Fabric evaluation lock")
    manifest = read_json(manifest_path())
    result = {
        "action": "seal",
        "snapshot_id": snapshot_id,
        "table_count": len(manifest.get("tables", [])),
    }
else:
    if not notebookutils.fs.exists(lock_path()):
        raise RuntimeError("No active Fabric evaluation snapshot lock exists")
    active = read_json(lock_path())
    if active.get("snapshot_id") != snapshot_id or active.get("status") != "sealed":
        raise RuntimeError(f"Snapshot {snapshot_id} does not own a sealed evaluation lock")
    manifest = read_json(manifest_path())
    if manifest.get("snapshot_id") != snapshot_id:
        raise RuntimeError("Fabric evaluation snapshot manifest ID does not match")
    restored: list[dict[str, object]] = []
    for raw_entry in manifest.get("tables", []):
        if not isinstance(raw_entry, dict):
            raise RuntimeError("Fabric evaluation snapshot contains an invalid table entry")
        lakehouse_id = str(raw_entry["lakehouse_id"])
        table_name = str(raw_entry["table"])
        version = int(raw_entry["version"])
        path = table_path(lakehouse_id, table_name)
        delta = DeltaTable.forPath(spark, path)
        delta.restoreToVersion(version)
        actual = fingerprint(DeltaTable.forPath(spark, path).toDF())
        if actual != raw_entry.get("fingerprint"):
            raise RuntimeError(f"Restored fingerprint mismatch for {raw_entry['lakehouse']}.{table_name}")
        restored.append(
            {
                "lakehouse": raw_entry["lakehouse"],
                "table": table_name,
                "version": version,
                "fingerprint": actual,
            }
        )
    remove_required(manifest_path())
    remove_required(lock_path())
    if notebookutils.fs.exists(manifest_path()) or notebookutils.fs.exists(lock_path()):
        raise RuntimeError("Fabric evaluation cleanup paths still exist after removal")
    result = {
        "action": "restore",
        "snapshot_id": snapshot_id,
        "verified": True,
        "table_count": len(restored),
        "tables": restored,
    }

notebookutils.notebook.exit(json.dumps(result, default=str, sort_keys=True))